# From clause to formula: a logician's path

> **Demonstration only:** frozen synthetic data, not evidence about any real decision. This notebook opens the property language and its documented refusals; it does not provide legal advice.

This is a runnable path through the formal language: a clause, its authored `spec`, the parser and atom inventory, the narrowest fragment, and a result from a shipped synthetic rule system. The mathematical definitions remain in [`docs/theory/02-syntax.md`](../docs/theory/02-syntax.md) and [`docs/theory/03-semantics.md`](../docs/theory/03-semantics.md); this notebook only makes the objects visible. The correspondence between claims and tests is recorded in [`docs/theory/claim-map.md`](../docs/theory/claim-map.md).

In [1]:
import ast
from dataclasses import replace
from pathlib import Path

from reasonsmith import rulelang
from reasonsmith.examples import symbolic_rules
from reasonsmith.manyvalued import ALGEBRAS
from reasonsmith.report import check_conformance
from reasonsmith.spec import load_pack


def _refinement_record():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        record = candidate / "docs" / "refinement.md"
        if record.exists():
            return record
    raise FileNotFoundError(
        "docs/refinement.md was not found above the working directory: "
        "this notebook reads the committed refinement record, so it needs a repository checkout"
    )


REFINEMENT_RECORD = _refinement_record()


def refinement_row(requirement_id):
    lines = REFINEMENT_RECORD.read_text().splitlines()
    return next(line for line in lines if f"`{requirement_id}` |" in line and line.startswith("|"))


def result_head(report):
    result = report.to_dict()["results"][0]
    return {key: result[key] for key in ("verdict", "strength", "basis")}

pack = load_pack("ecoa")
system = symbolic_rules.system_under_test()
print("Loaded pack:", pack.id)
print("System:", type(system).__name__)

Loaded pack: ecoa
System: RulesAdapter


## Walk 1 — a record property

The first row is GDPR Article 22(3). Its fourth column is the boundary: the formula uses two named record fields as proxies, but it does not turn *suitable* into a predicate. That distinction is the record fragment described by Definition 2.6 in [`docs/theory/02-syntax.md`](../docs/theory/02-syntax.md). The row below is copied from the repository's refinement record, including the omitted material; this is the source-backed limitation, not a gloss. [`docs/theory/claim-map.md`](../docs/theory/claim-map.md) links the presence semantics to `test_the_solvers_blank_string_is_pythons_blank_string`.

In [2]:
record_requirement = load_pack("gdpr").get_requirement(
    "gdpr_art22_3_safeguards_human_intervention"
)
print("CLAUSE (verbatim_text):")
print(record_requirement.verbatim_text)
print("\nREFINEMENT ROW (all four columns):")
print(refinement_row(record_requirement.id))

CLAUSE (verbatim_text):
In the cases referred to in points (a) and (c) of paragraph 2, the data controller shall implement suitable measures to safeguard the data subject's rights and freedoms and legitimate interests, at least the right to obtain human intervention on the part of the controller, to express his or her point of view and to contest the decision.

REFINEMENT ROW (all four columns):
| GDPR Article 22(3)<br>`gdpr_art22_3_safeguards_human_intervention` | The controller must implement suitable measures safeguarding the data subject — at least human intervention, expressing a point of view, and contesting the decision. | `record`: `present(artifact_logs_decision_record) and present(scope_statements_local_vs_global)` | All three named rights. Neither conjunct witnesses human intervention, a point of view or a contest; `scope_statements_local_vs_global` is a statement of explanation scope, chosen by the pack author as a proxy for *the safeguard can say what this decision rested 

### The authored surface and its rewrite

`spec` is executable property text; the English rationale lives in a separate field. Arrow syntax is surface sugar: [`rulelang.preprocess_spec`](../src/reasonsmith/rulelang.py) rewrites it before parsing, as specified in [`docs/theory/02-syntax.md`](../docs/theory/02-syntax.md). The rewrite behavior is pinned by `test_the_rewriter_never_collapses_equivalence_to_a_comparison` and the arrow tests in [`docs/theory/claim-map.md`](../docs/theory/claim-map.md).

In [3]:
record_spec = record_requirement.spec
print("AUTHORED spec:")
print(record_spec)
print("\nPREPROCESSED:")
print(rulelang.preprocess_spec(record_spec))

AUTHORED spec:
present(artifact_logs_decision_record) and present(scope_statements_local_vs_global)

PREPROCESSED:
present(artifact_logs_decision_record) and present(scope_statements_local_vs_global)


### Parse, then name the atoms

The parser returns Python's expression AST after the language whitelist has accepted it. The atom inventory is the useful bridge from formula to record: the signal names inside `present(...)` are the propositional letters that the engines bind to fields. [`docs/theory/02-syntax.md`](../docs/theory/02-syntax.md) Definition 2.2 gives the call forms and Definition 2.4 their side conditions; the solver/interpreter agreement is pinned by `test_the_encoder_and_the_interpreter_answer_the_same`.

In [4]:
record_ast = rulelang.parse_property(record_spec)
print(ast.dump(record_ast, indent=2))
print("\npresence_atoms:", rulelang.presence_atoms(record_ast))
print("degree_atoms:", rulelang.degree_atoms(record_ast))
print("undetermined_atoms:", rulelang.undetermined_atoms(record_ast))
print("counterfactual_atom:", rulelang.counterfactual_atom(record_ast))
print("statistical_atom:", rulelang.statistical_atom(record_ast))

Expression(
  body=BoolOp(
    op=And(),
    values=[
      Call(
        func=Name(id='present', ctx=Load()),
        args=[
          Name(id='artifact_logs_decision_record', ctx=Load())],
        keywords=[]),
      Call(
        func=Name(id='present', ctx=Load()),
        args=[
          Name(id='scope_statements_local_vs_global', ctx=Load())],
        keywords=[])]))

presence_atoms: ('artifact_logs_decision_record', 'scope_statements_local_vs_global')
degree_atoms: ()
undetermined_atoms: ()
counterfactual_atom: None
statistical_atom: None


### Fragment and evaluation

`classify_fragment` chooses the narrowest fragment, rather than a looser compatible label. The fragment controls which engines may discharge the property; Definition 2.6 in [`docs/theory/02-syntax.md`](../docs/theory/02-syntax.md) gives the order. Here the shipped symbolic rule system exposes the rule set, so the result reaches the proved rung. The result below deliberately prints only the verdict, rung, and evidence basis. [`docs/theory/08-evidence.md`](../docs/theory/08-evidence.md) defines those evidence coordinates.

In [5]:
print("classify_fragment:", rulelang.classify_fragment(record_spec))
record_report = check_conformance(
    system,
    replace(load_pack("gdpr"), requirements=(record_requirement,)),
)
print("result:", result_head(record_report))

classify_fragment: record
result: {'verdict': 'satisfied', 'strength': 'proved', 'basis': 'behavioural'}


### What this formalisation cannot say

The fourth column says exactly what is absent: no field witnesses human intervention, a point of view, a contest, or the legal adequacy of a safeguard; the scope field is only the pack's chosen proxy, and the property is applied to every trace record. This is the refusal boundary, not a hidden claim. See [`docs/theory/03-semantics.md`](../docs/theory/03-semantics.md) Definition 3.5 and the row above; the shipped-row boundary is kept honest by the refinement tests named in [`docs/theory/claim-map.md`](../docs/theory/claim-map.md).

## Walk 2 — a temporal property

The second row has the same first step but a different shape: an obligation starts at a notice and ends at one of two recorded events. Its fourth column explicitly refuses to invent the duration or inspect the notice's text. The temporal operators and their finite-trace reading are Definitions 2.6 and 3.8 in [`docs/theory/02-syntax.md`](../docs/theory/02-syntax.md) and [`docs/theory/03-semantics.md`](../docs/theory/03-semantics.md). `test_the_ltlf_backend_agrees_with_the_monitor` and `test_only_always_reaches_the_temporal_proof_rung` are the relevant claim-map checks.

In [6]:
temporal_requirement = pack.get_requirement(
    "ecoa_reg_b_1002_9_a_1_timing_of_notice"
)
print("CLAUSE (verbatim_text):")
print(temporal_requirement.verbatim_text)
print("\nREFINEMENT ROW (all four columns):")
print(refinement_row(temporal_requirement.id))

CLAUSE (verbatim_text):
A creditor shall notify an applicant of action taken within:
(i) 30 days after receiving a completed application concerning the creditor's approval of, counteroffer to, or adverse action on the application;
(ii) 30 days after taking adverse action on an incomplete application, unless notice is provided in accordance with paragraph (c) of this section;
(iii) 30 days after taking adverse action on an existing account; or
(iv) 90 days after notifying the applicant of a counteroffer if the applicant does not expressly accept or use the credit offered.

REFINEMENT ROW (all four columns):
| 12 CFR 1002.9(a)(1)<br>`ecoa_reg_b_1002_9_a_1_timing_of_notice` | A creditor must notify an applicant of action taken within 30 days of a completed application, an incomplete application, or an existing account — or within 90 days of a counteroffer the applicant did not accept. | `temporal`: `always(present(artifact_logs_decision_record) -> ((artifact_logs_notification_latency_days

### Surface, AST, and atoms

The authored arrow is again rewritten into an `Implies(...)` call. `always(...)` and the disjunction remain visible in the AST; no temporal meaning is implemented by this notebook. The shared parser and atom extractors are the public path described in [`docs/theory/02-syntax.md`](../docs/theory/02-syntax.md). Read `presence_atoms: None` below as *this formula is not a conjunction of `present()` atoms* — the shape the record engine specialises — and not as an absence of `present(...)`, which the AST above plainly contains.

In [7]:
temporal_spec = temporal_requirement.spec
print("AUTHORED spec:")
print(temporal_spec)
print("\nPREPROCESSED:")
print(rulelang.preprocess_spec(temporal_spec))

temporal_ast = rulelang.parse_property(temporal_spec)
print("\nAST:")
print(ast.dump(temporal_ast, indent=2))
print("\npresence_atoms:", rulelang.presence_atoms(temporal_ast))
print("degree_atoms:", rulelang.degree_atoms(temporal_ast))
print("undetermined_atoms:", rulelang.undetermined_atoms(temporal_ast))
print("counterfactual_atom:", rulelang.counterfactual_atom(temporal_ast))
print("statistical_atom:", rulelang.statistical_atom(temporal_ast))

AUTHORED spec:
always(present(artifact_logs_decision_record) -> ((artifact_logs_notification_latency_days <= 30) or ((artifact_logs_counteroffer_not_accepted >= 0.5) and (artifact_logs_notification_latency_days <= 90))))

PREPROCESSED:
always(Implies((present(artifact_logs_decision_record) ), ( ((artifact_logs_notification_latency_days <= 30) or ((artifact_logs_counteroffer_not_accepted >= 0.5) and (artifact_logs_notification_latency_days <= 90))))))

AST:
Expression(
  body=Call(
    func=Name(id='always', ctx=Load()),
    args=[
      Call(
        func=Name(id='Implies', ctx=Load()),
        args=[
          Call(
            func=Name(id='present', ctx=Load()),
            args=[
              Name(id='artifact_logs_decision_record', ctx=Load())],
            keywords=[]),
          BoolOp(
            op=Or(),
            values=[
              Compare(
                left=Name(id='artifact_logs_notification_latency_days', ctx=Load()),
                ops=[
                  LtE(

### Fragment, evaluation, and refusal boundary

The narrowest fragment is now `temporal`, so the ladder can use a temporal reduction when the exposed logic permits it. The symbolic example proves this property over its declared input constraints; the output still reports only the three fields relevant to this walk. The fourth column is the cost: this formula reads flags in the trace, not the reasonableness of the period, the notice wording, or application identity across interleaved records. The rung reached here is the solver's, so [`docs/theory/05-decision-procedures.md`](../docs/theory/05-decision-procedures.md) §5.2 states the procedure, while [`docs/theory/03-semantics.md`](../docs/theory/03-semantics.md) owns the finite-trace interpretation.

In [8]:
print("classify_fragment:", rulelang.classify_fragment(temporal_spec))
temporal_report = check_conformance(
    system,
    replace(pack, requirements=(temporal_requirement,)),
)
print("result:", result_head(temporal_report))

classify_fragment: temporal
result: {'verdict': 'satisfied', 'strength': 'proved', 'basis': 'behavioural'}


## Payoff: three algebras, three conjunctions

Many-valued formulas are read over a declared algebra, not a presentation setting. The same two half-degrees are combined differently by Łukasiewicz, Gödel, and product. This is the semantic commitment described in [`docs/theory/08-evidence.md`](../docs/theory/08-evidence.md) §8.5 and [`docs/theory/03-semantics.md`](../docs/theory/03-semantics.md) Definitions 3.2 and 3.3; the degree over a trace is Definition 3.12 there. This layer is not the Kleene chain of Definition 3.11, which is ignorance about a record rather than a truth degree. `test_the_three_algebras_disagree_about_a_conjunction_of_two_halves` pins the difference.

In [9]:
left = right = 0.5
print("conjunction of two half-degrees:")
for name, algebra in ALGEBRAS.items():
    print(f"{name:12} -> {algebra.conjunction(left, right):.2f}")
print("\nA pack declaring its t-norm is choosing this semantics, not choosing formatting.")

conjunction of two half-degrees:
lukasiewicz  -> 0.00
godel        -> 0.50
product      -> 0.25

A pack declaring its t-norm is choosing this semantics, not choosing formatting.


## A refused construct

The monitor backend has four known divergence shapes in Remark 3.3 of [`docs/theory/03-semantics.md`](../docs/theory/03-semantics.md); the remainder operator is one of the refused shapes. A property using it is rejected with the construct named, rather than silently rendered under a different reading. This is pinned by `test_a_duty_using_a_misread_shape_is_not_evaluated_and_names_the_construct` and `test_the_four_named_shapes_are_still_what_the_document_records` in [`docs/theory/claim-map.md`](../docs/theory/claim-map.md).

In [10]:
from reasonsmith.engines.observed import to_stl

refused_spec = "artifact_logs_notification_latency_days % 2 == 0"
print("spec:", refused_spec)
try:
    to_stl(refused_spec)
except Exception as error:
    print(type(error).__name__ + ":", error)

spec: artifact_logs_notification_latency_days % 2 == 0
MisreadShapeError: the remainder operator `%`: rtamt's lexer has no `%` and ANTLR error-recovers by dropping the token instead of raising, so the monitor would answer about a formula nobody wrote


The two walks show the boundary in both directions: the parser exposes the letters and structure, while the fragment and evidence path decide what can be established. For the formal definitions and their tests, continue through [`docs/theory/00-notation.md`](../docs/theory/00-notation.md), [`docs/theory/03-semantics.md`](../docs/theory/03-semantics.md), and [`docs/theory/claim-map.md`](../docs/theory/claim-map.md).